# Earthquake deaths, magnitude and income: World Bank

**Question:** after allowing for magnitude and depth, how much does a country's income explain earthquake deaths?

This notebook collects annual country/entity indicators from the World Bank's World Development Indicators (WDI) for 2000–2025. The main World Bank request includes nominal and real/PPP income indicators, population and exposure, development context, **Gini inequality**, and **adjusted net savings**.

- Raw API/source responses are saved unchanged in `data/raw/`.
- Derived tables go to `data/interim/`.
- Aggregates (World, Sub-Saharan Africa, income groups, ...) are **flagged, not dropped** here. Selecting the final country list belongs in the cleaning notebook.
- Genuine Progress Indicator (GPI) is not presented as a World Bank WDI indicator. Adjusted net savings is included as a distinct sustainability-related measure, not labelled GPI.
- Missing values remain missing; this notebook does not impute or silently forward-fill them.


## 1. Setup and project folders

In [12]:
# Standard library
from pathlib import Path
from datetime import datetime
from urllib.robotparser import RobotFileParser
import os
import json
import time

# External libraries
import requests
from requests.exceptions import RequestException
import pandas as pd

# Project directories
PROJECT_DIR = Path(os.getcwd())

RAW_DIR = PROJECT_DIR / "raw"
INTERIM_DIR = PROJECT_DIR / "interim"
CLEAN_DIR = PROJECT_DIR / "clean"
REPORT_DIR = PROJECT_DIR / "reports"
WB_YEARLY_DIR = INTERIM_DIR / "worldbank_yearly_csv"

for folder in [RAW_DIR, INTERIM_DIR, CLEAN_DIR, REPORT_DIR, WB_YEARLY_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Descriptive user agent
USER_AGENT = "EarthquakeAcademicProject/1.0 (contact: saivarshithchilukuri@gmail.com)"

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

WB_API = "https://api.worldbank.org/v2"

print("Collection date: 2026-10-22")
print("Project folders ready under:", PROJECT_DIR)

Collection date: 2026-10-22
Project folders ready under: d:\Pandu\IIT_Madras\DA2402\DA2402_Project_Earthquakes


## 2. robots.txt and terms of use

The robots response is saved as evidence. Quote the printed lines and the licence in the report.

In [13]:
WB_ROBOTS_URL = "https://api.worldbank.org/robots.txt"
WB_CHECK_URL = f"{WB_API}/country/all/indicator/NY.GDP.PCAP.CD"

robots_response = session.get(WB_ROBOTS_URL, timeout=30, allow_redirects=True)

robots_path = REPORT_DIR / f"robots_worldbank.txt"

robots_path.write_text(
    f"Requested URL: {WB_ROBOTS_URL}\n"
    f"Final URL: {robots_response.url}\n"
    f"HTTP status: {robots_response.status_code}\n\n"
    + robots_response.text,
    encoding="utf-8",
)

print("Robots URL:", WB_ROBOTS_URL)
print("HTTP status:", robots_response.status_code)
print("Saved evidence:", robots_path)
print("\n--- robots.txt (first 25 lines, quote in report) ---")
print("\n".join(robots_response.text.splitlines()[:25]))

if robots_response.status_code == 200:
    parser = RobotFileParser()
    parser.parse(robots_response.text.splitlines())
    robots_allowed = parser.can_fetch(USER_AGENT, WB_CHECK_URL)
    print("\nAllowed by robots rules:", robots_allowed)
    if not robots_allowed:
        raise RuntimeError("The API endpoint is disallowed by robots.txt.")
elif robots_response.status_code == 404:
    print("\nrobots.txt returned 404. Review the API terms separately.")
else:
    raise RuntimeError(f"Robots audit needs manual review. HTTP status: {robots_response.status_code}")

WORLD_BANK_TERMS_URL = "https://www.worldbank.org/ext/en/legal/terms-conditions"
WORLD_BANK_DATA_TERMS_URL = "https://data.worldbank.org/summary-terms-of-use"
print("\nAPI terms:", WORLD_BANK_TERMS_URL)
print("Dataset terms (CC BY 4.0 for most WDI data; confirm on this page):", WORLD_BANK_DATA_TERMS_URL)

Robots URL: https://api.worldbank.org/robots.txt
HTTP status: 404
Saved evidence: d:\Pandu\IIT_Madras\DA2402\DA2402_Project_Earthquakes\reports\robots_worldbank.txt

--- robots.txt (first 25 lines, quote in report) ---
{ "statusCode": 404, "message": "Resource not found" }

robots.txt returned 404. Review the API terms separately.

API terms: https://www.worldbank.org/ext/en/legal/terms-conditions
Dataset terms (CC BY 4.0 for most WDI data; confirm on this page): https://data.worldbank.org/summary-terms-of-use


In [14]:
WB_ROBOTS_NONAPI_URL="https://worldbank.org/robots.txt"
robots_nonapi_response = session.get(WB_ROBOTS_NONAPI_URL, timeout=30, allow_redirects=True)

ROBOTS_NONAPI_PATH= REPORT_DIR / f"robots_worldbank_nonapi_2026-10-22.txt"

ROBOTS_NONAPI_PATH.write_text(
    f"Requested URL: {WB_ROBOTS_NONAPI_URL}\n"
    f"Final URL: {robots_nonapi_response.url}\n"
    f"HTTP status: {robots_nonapi_response.status_code}\n\n"
    + robots_nonapi_response.text,
    encoding="utf-8",
)

print("Robots URL:", WB_ROBOTS_NONAPI_URL)
print("HTTP status:", robots_nonapi_response.status_code)
print("Saved evidence:", ROBOTS_NONAPI_PATH)
print("\n--- robots.txt (first 25 lines, quote in report) ---")
print("\n".join(robots_nonapi_response.text.splitlines()[:25]))

if robots_nonapi_response.status_code == 200:
    parser = RobotFileParser()
    parser.parse(robots_nonapi_response.text.splitlines())
    robots_allowed = parser.can_fetch(USER_AGENT, WB_CHECK_URL)
    print("\nAllowed by robots rules:", robots_allowed)
    if not robots_allowed:
        raise RuntimeError("The API endpoint is disallowed by robots.txt.")
elif robots_nonapi_response.status_code == 404:
    # A 404 means no robots file; it is not a terms-of-use approval.
    print("\nrobots.txt returned 404. Review the API terms separately.")
else:
    raise RuntimeError(f"Robots audit needs manual review. HTTP status: {robots_nonapi_response.status_code}")

WORLD_BANK_TERMS_URL = "https://www.worldbank.org/ext/en/legal/terms-conditions"
WORLD_BANK_DATA_TERMS_URL = "https://data.worldbank.org/summary-terms-of-use"
print("\nAPI terms:", WORLD_BANK_TERMS_URL)
print("Dataset terms (CC BY 4.0 for most WDI data; confirm on this page):", WORLD_BANK_DATA_TERMS_URL)




Robots URL: https://worldbank.org/robots.txt
HTTP status: 200
Saved evidence: d:\Pandu\IIT_Madras\DA2402\DA2402_Project_Earthquakes\reports\robots_worldbank_nonapi_2026-10-22.txt

--- robots.txt (first 25 lines, quote in report) ---
User-agent: *
Allow: /

# --- Adobe Edge Optimize crawler ---
User-agent: AdobeEdgeOptimize/1.0
Allow: /

# --- System / platform paths ---
Disallow: /apps/
Disallow: /conf/
Disallow: /misc/
Disallow: /wbg/services/
Disallow: /wbg/aem/
Disallow: /fragments/
Disallow: /fragment/
Disallow: /downloadstats/
Disallow: /content/search
Disallow: /*/search
Disallow: /*/webarchives/archive
Disallow: /*.print

# --- One-off retired pages ---
Disallow: /*/programs/umbrellafacilityforgenderequality/brief/ufge-partnership-council-meeting

# --- Retired templates: blocked entirely, all years, all locales ---

Allowed by robots rules: True

API terms: https://www.worldbank.org/ext/en/legal/terms-conditions
Dataset terms (CC BY 4.0 for most WDI data; confirm on this page):

## 3. Indicators to collect

`inflation_adjusted = True` in the indicator metadata marks values expressed in constant-price or PPP-constant terms; it does **not** mean every indicator with an inflation-related name should be True. For example, the GDP deflator is itself an annual price-change rate, while current-US-dollar GDP is nominal.

The main World Bank indicator list includes inequality (`SI.POV.GINI`) and sustainability-accounting data (`NY.ADJ.SVNG.GN.ZS`) so they are collected with the other WDI indicators in the **same API workflow**. HDI is maintained by UNDP, not WDI, so it is downloaded from UNDP separately and then joined into a combined country-year panel by ISO3 code and year.

Use a constant-price or PPP-constant income series (suggested: `gdp_per_capita_ppp_constant_2021`) for real cross-country comparisons. Keep nominal-current-US$ measures separate because they answer a different question. Avoid putting every column into one regression: GDP = GDP per capita × population, and urban + rural = total population. Health expenditure and electricity access can be context or robustness variables rather than automatically treated as independent causes.


In [15]:
START_YEAR = 2000
END_YEAR = 2025
MAX_ATTEMPTS = 3
PER_PAGE = 20000
WB_SOURCE_ID = 2  # World Development Indicators

# code: (column name, description, inflation_adjusted)
WB_INDICATORS = {
    # Income and economic size: current US$ (NOT inflation-adjusted)
    "NY.GDP.MKTP.CD":    ("gdp_current_usd", "GDP, current US$", False),
    "NY.GDP.PCAP.CD":    ("gdp_per_capita_current_usd", "GDP per capita, current US$", False),

    # Income and economic size: inflation-adjusted (constant prices / PPP)
    "NY.GDP.MKTP.KD":    ("gdp_constant_2015_usd", "GDP, constant 2015 US$", True),
    "NY.GDP.PCAP.KD":    ("gdp_per_capita_constant_2015_usd", "GDP per capita, constant 2015 US$", True),
    "NY.GNP.PCAP.KD":    ("gni_per_capita_constant_2015_usd", "GNI per capita, constant 2015 US$", True),
    "NY.GDP.MKTP.PP.KD": ("gdp_ppp_constant_2021", "GDP, PPP, constant 2021 international $", True),
    "NY.GDP.PCAP.PP.KD": ("gdp_per_capita_ppp_constant_2021", "GDP per capita, PPP, constant 2021 international $", True),
    "NY.GNP.PCAP.PP.KD": ("gni_per_capita_ppp_constant_2021", "GNI per capita, PPP, constant 2021 international $", True),

    # Growth and prices
    "NY.GDP.MKTP.KD.ZG": ("real_gdp_growth_pct", "Real GDP growth, annual %", False),
    "NY.GDP.PCAP.KD.ZG": ("real_gdp_per_capita_growth_pct", "Real GDP per capita growth, annual %", False),
    "FP.CPI.TOTL.ZG":    ("inflation_cpi_pct", "Inflation, consumer prices, annual %", False),
    "NY.GDP.DEFL.KD.ZG": ("inflation_gdp_deflator_pct", "Inflation, GDP deflator, annual %", False),

    # Population and exposure
    "SP.POP.TOTL":       ("population_total", "Population, total", False),
    "EN.POP.DNST":       ("population_density_per_sq_km", "Population density, people per sq km", False),
    "SP.POP.GROW":       ("population_growth_pct", "Population growth, annual %", False),
    "SP.URB.TOTL":       ("urban_population", "Urban population", False),
    "SP.URB.TOTL.IN.ZS": ("urban_population_pct", "Urban population, % of total", False),
    "SP.RUR.TOTL":       ("rural_population", "Rural population", False),
    "EN.URB.MCTY.TL.ZS": ("pop_in_large_cities_pct", "Population in urban agglomerations >1 million, % of total", False),
    "AG.LND.TOTL.K2":    ("land_area_sq_km", "Land area, sq km", False),
    "SP.POP.65UP.TO.ZS": ("pop_65_plus_pct", "Population ages 65 and above, % of total", False),

    # Inequality and sustainability-related context (also available in WDI)
    # Gini uses survey-based observations; coverage and comparability are irregular.
    "SI.POV.GINI":       ("gini_index", "Gini index (income/consumption inequality; 0-100)", False),
    # Adjusted net savings is related to sustainability accounting, but is NOT GPI.
    "NY.ADJ.SVNG.GN.ZS": ("adjusted_net_savings_pct_gni", "Adjusted net savings incl. particulate-emission damage, % of GNI", False),

    # Development and vulnerability context
    "EG.ELC.ACCS.ZS":    ("electricity_access_pct", "Access to electricity, % of population", False),
    "SH.XPD.CHEX.GD.ZS": ("health_expenditure_pct_gdp", "Current health expenditure, % of GDP", False),
    "SL.UEM.TOTL.ZS":    ("unemployment_pct", "Unemployment, % of labour force", False),
    "SH.MED.PHYS.ZS":    ("physicians_per_1000", "Physicians, per 1,000 people", False),
    "SH.MED.BEDS.ZS":    ("hospital_beds_per_1000", "Hospital beds, per 1,000 people", False),
    "SP.DYN.LE00.IN":    ("life_expectancy_years", "Life expectancy at birth, years", False),
    "EN.POP.SLUM.UR.ZS": ("slum_population_pct_urban", "Population living in slums, % of urban population", False),
}
print("Indicators requested:", len(WB_INDICATORS))

Indicators requested: 30


### 3a. Validate indicator codes and save their metadata

One call per indicator. This catches a wrong code before the big download (the API answers HTTP 200 with an error message for bad codes) and gives the name, source and description needed for the data dictionary.

In [16]:
indicator_payloads = {}
meta_rows = []
valid_codes = []

for code, (col, desc, adjusted) in WB_INDICATORS.items():
    try:
        r = session.get(f"{WB_API}/indicator/{code}", params={"format": "json", "source": WB_SOURCE_ID}, timeout=(15, 60))
        r.raise_for_status()
        p = r.json()
    except (RequestException, ValueError) as exc:
        print(f"{code}: request failed ({exc!r}); skipping")
        continue

    ok = isinstance(p, list) and len(p) > 1 and p[1]
    if not ok:
        print(f"{code}: NOT FOUND in source {WB_SOURCE_ID}; skipping ->", p[0] if p else p)
        continue

    info = p[1][0]
    indicator_payloads[code] = p
    valid_codes.append(code)
    meta_rows.append({
        "indicator_code": code,
        "column": col,
        "wb_name": info.get("name"),
        "source": (info.get("source") or {}).get("value"),
        "organization": info.get("sourceOrganization"),
        "inflation_adjusted": adjusted,
        "definition": info.get("sourceNote"),
    })
    time.sleep(0.001)

(RAW_DIR / f"worldbank_indicator_metadata_2026-10-22.json").write_text(
    json.dumps(indicator_payloads, ensure_ascii=False, indent=2), encoding="utf-8")

indicator_dictionary = pd.DataFrame(meta_rows)  # [n_valid, 7]
indicator_dictionary.to_csv(INTERIM_DIR / f"worldbank_indicator_dictionary.csv", index=False)

COL_BY_CODE = {c: WB_INDICATORS[c][0] for c in valid_codes}
WB_URL = f"{WB_API}/country/all/indicator/" + ";".join(valid_codes)

print("Valid indicators:", len(valid_codes), "of", len(WB_INDICATORS))
display(indicator_dictionary[["indicator_code", "column", "wb_name", "inflation_adjusted"]])

Valid indicators: 30 of 30


,indicator_code,column,wb_name,inflation_adjusted
0,NY.GDP.MKTP.CD,gdp_current_usd,GDP (current US$),False
1,NY.GDP.PCAP.CD,gdp_per_capita_current_usd,GDP per capita (current US$),False
2,NY.GDP.MKTP.KD,gdp_constant_2015_usd,GDP (constant 2015 US$),True
3,NY.GDP.PCAP.KD,gdp_per_capita_constant_2015_usd,GDP per capita (constant 2015 US$),True
4,NY.GNP.PCAP.KD,gni_per_capita_constant_2015_usd,GNI per capita (constant 2015 US$),True
5,NY.GDP.MKTP.PP.KD,gdp_ppp_constant_2021,"GDP, PPP (constant 2021 international $)",True
6,NY.GDP.PCAP.PP.KD,gdp_per_capita_ppp_constant_2021,"GDP per capita, PPP (constant 2021 internation...",True
7,NY.GNP.PCAP.PP.KD,gni_per_capita_ppp_constant_2021,"GNI per capita, PPP (constant 2021 internation...",True
8,NY.GDP.MKTP.KD.ZG,real_gdp_growth_pct,GDP growth (annual %),False
9,NY.GDP.PCAP.KD.ZG,real_gdp_per_capita_growth_pct,GDP per capita growth (annual %),False


## 4. Country metadata (region, income group, aggregate flag)

`country/all` returns real countries and aggregates together. The country endpoint tells them apart: aggregates have region "Aggregates". Region and income group are also what the region-wise plots will group by.

In [17]:
r = session.get(f"{WB_API}/country", params={"format": "json", "per_page": 400}, timeout=(15, 60))
r.raise_for_status()
assert r.json()[0]["pages"] == 1, "Country list spans more than one page; raise per_page."

countries_raw_path = RAW_DIR / f"worldbank_countries.json"
countries_raw_path.write_bytes(r.content)  # raw, unedited

c = pd.json_normalize(r.json()[1])
countries = pd.DataFrame({
    "country_iso3": c["id"],
    "country_iso2": c["iso2Code"],
    "country_name": c["name"],
    "region": c["region.value"].str.strip(),
    "income_group": c["incomeLevel.value"].str.strip(),
})  # [~296, 5]
countries["is_aggregate"] = countries["region"].eq("Aggregates")

countries.to_csv(INTERIM_DIR / f"worldbank_country_metadata_2026-10-22.csv", index=False)

print("Entities:", len(countries), "| real countries:", int((~countries["is_aggregate"]).sum()),
      "| aggregates:", int(countries["is_aggregate"].sum()))
print("\nRegions:\n", countries.loc[~countries["is_aggregate"], "region"].value_counts().to_string())

Entities: 296 | real countries: 217 | aggregates: 79

Regions:
 region
Europe & Central Asia                                58
Sub-Saharan Africa                                   48
Latin America & Caribbean                            42
East Asia & Pacific                                  37
Middle East, North Africa, Afghanistan & Pakistan    23
South Asia                                            6
North America                                         3


## 5. Download all indicators, one request per year

In [18]:
class WBApiError(Exception):
    """API answered HTTP 200 with an error message; retrying will not help."""

all_records = []
request_log = []


def fetch_wb_page(year, page):
    params = {"date": str(year), "format": "json", "per_page": PER_PAGE,
              "page": page, "source": WB_SOURCE_ID}

    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            response = session.get(WB_URL, params=params, timeout=(15, 120))
            response.raise_for_status()
            payload = response.json()

            if isinstance(payload, list) and payload and isinstance(payload[0], dict) and "message" in payload[0]:
                raise WBApiError(str(payload[0]["message"]))
            if not isinstance(payload, list) or len(payload) < 2 or not isinstance(payload[0], dict):
                raise ValueError(f"Unexpected API response for {year}, page {page}")

            records_page = payload[1] or []

            # Preserve the original API response unchanged.
            raw_path = RAW_DIR / f"worldbank_indicators_{year}_page_{page:03d}.json"
            raw_path.write_bytes(response.content)

            return payload[0], records_page, response.url, raw_path

        except WBApiError:
            raise
        except (RequestException, ValueError) as exc:
            print(f"{year}, page {page}, attempt {attempt}/{MAX_ATTEMPTS}: {exc}")
            if attempt < MAX_ATTEMPTS:
                time.sleep(2 * attempt)

    raise RuntimeError(f"Failed to retrieve {year}, page {page}")


for year in range(START_YEAR, END_YEAR + 1):
    metadata, page_records, request_url, raw_path = fetch_wb_page(year, 1)
    all_records.extend(page_records)
    request_log.append({"year": year, "page": 1, "url": request_url,
                        "records": len(page_records), "raw_file": str(raw_path)})

    for page in range(2, int(metadata.get("pages", 1)) + 1):
        _, page_records, request_url, raw_path = fetch_wb_page(year, page)
        all_records.extend(page_records)
        request_log.append({"year": year, "page": page, "url": request_url,
                            "records": len(page_records), "raw_file": str(raw_path)})
        time.sleep(0.002)

    print(f"{year}: completed; total records = {len(all_records)}")
    time.sleep(0.002)

if not all_records:
    raise RuntimeError("No records collected.")

# Combined observations (derived from the raw pages) in interim/, with source and date in the name.
combined_path = INTERIM_DIR / f"worldbank_all_indicators_{START_YEAR}_{END_YEAR}.json"
with open(combined_path, "w", encoding="utf-8") as f:
    json.dump(all_records, f, ensure_ascii=False, indent=2)

log_path = REPORT_DIR / f"worldbank_request_log_2026-10-22.csv"
pd.DataFrame(request_log).to_csv(log_path, index=False)

returned_codes = {(rec.get("indicator") or {}).get("id") for rec in all_records}
print("\nCollection finished.")
print("Total records:", len(all_records))
print("Requested (valid) indicators:", len(valid_codes), "| returned:", len(returned_codes))
print("Combined JSON:", combined_path)
print("Request log:", log_path)
print("Missing from response:", sorted(set(valid_codes) - returned_codes))

2000: completed; total records = 7980
2001: completed; total records = 15960
2002: completed; total records = 23940
2003: completed; total records = 31920
2004: completed; total records = 39900
2005: completed; total records = 47880
2006: completed; total records = 55860
2007: completed; total records = 63840
2008: completed; total records = 71820
2009: completed; total records = 79800
2010: completed; total records = 87780
2011: completed; total records = 95760
2012: completed; total records = 103740
2013: completed; total records = 111720
2014: completed; total records = 119700
2015: completed; total records = 127680
2016: completed; total records = 135660
2017: completed; total records = 143640
2018: completed; total records = 151620
2019: completed; total records = 159600
2020: completed; total records = 167580
2021: completed; total records = 175560
2022: completed; total records = 183540
2023: completed; total records = 191520
2024: completed; total records = 199500
2025: complet

## 6. Inspect the raw data before any change

Shape, dtypes and the meaning of every kept column, as the data-collection component requires.

In [19]:
raw_df = pd.json_normalize(all_records)  # [n_records, ~10]
print("Shape:", raw_df.shape)
print("\nDtypes:\n", raw_df.dtypes.to_string())

COLUMN_MEANING = {
    "indicator.id": "World Bank indicator code (kept)",
    "indicator.value": "Indicator name, includes unit and price base (kept for checking)",
    "country.id": "World Bank 2-letter entity code; also used for aggregates (kept, join key)",
    "country.value": "Entity name as spelled by the API (kept for checking)",
    "countryiso3code": "ISO3 code; may be empty for some entities (not trusted, replaced from country metadata)",
    "date": "Calendar year as string (kept, converted to int)",
    "value": "Observation; null when no data is published (kept)",
    "unit": "Unit field; usually empty (dropped)",
    "obs_status": "Observation status flag; usually empty (dropped)",
    "decimal": "Display precision hint (dropped)",
}
display(pd.DataFrame({"column": list(COLUMN_MEANING), "meaning": list(COLUMN_MEANING.values()),
                      "present": [k in raw_df.columns for k in COLUMN_MEANING]}))
display(raw_df.head())
print("Null 'value' share (no data published):", round(raw_df["value"].isna().mean(), 3))

Shape: (207480, 11)

Dtypes:
 countryiso3code        str
date                   str
value              float64
unit                   str
obs_status             str
decimal              int64
indicator.id           str
indicator.value        str
country.id             str
country.value          str
scale                  str


,column,meaning,present
0,indicator.id,World Bank indicator code (kept),True
1,indicator.value,"Indicator name, includes unit and price base (...",True
2,country.id,World Bank 2-letter entity code; also used for...,True
3,country.value,Entity name as spelled by the API (kept for ch...,True
4,countryiso3code,ISO3 code; may be empty for some entities (not...,True
5,date,"Calendar year as string (kept, converted to int)",True
6,value,Observation; null when no data is published (k...,True
7,unit,Unit field; usually empty (dropped),True
8,obs_status,Observation status flag; usually empty (dropped),True
9,decimal,Display precision hint (dropped),True


,countryiso3code,date,value,unit,obs_status,decimal,indicator.id,indicator.value,country.id,country.value,scale
0,AFE,2000,2.870425e+11,,,0,NY.GDP.MKTP.CD,GDP (current US$),ZH,Africa Eastern and Southern,NaN
1,AFW,2000,1.427002e+11,,,0,NY.GDP.MKTP.CD,GDP (current US$),ZI,Africa Western and Central,NaN
2,ARB,2000,7.634001e+11,,,0,NY.GDP.MKTP.CD,GDP (current US$),1A,Arab World,NaN
3,CSS,2000,2.562554e+10,,,0,NY.GDP.MKTP.CD,GDP (current US$),S3,Caribbean small states,NaN
4,CEB,2000,4.208239e+11,,,0,NY.GDP.MKTP.CD,GDP (current US$),B8,Central Europe and the Baltics,NaN


Null 'value' share (no data published): 0.175


## 7. Convert to yearly wide CSVs and one combined panel

One row per entity-year, one column per indicator. Region, income group and `is_aggregate` are attached. Nothing is dropped.

In [20]:
long_df = pd.DataFrame({
    "country_iso2": raw_df["country.id"],
    "year": pd.to_numeric(raw_df["date"], errors="coerce"),
    "code": raw_df["indicator.id"],
    "value": raw_df["value"],
}).dropna(subset=["year", "code"])  # [n_records, 4]
long_df["year"] = long_df["year"].astype(int)
long_df["column"] = long_df["code"].map(COL_BY_CODE)

wide = (long_df.set_index(["country_iso2", "year", "column"])["value"]
        .unstack("column")                       # [n_entity_years, n_indicators]
        .reindex(columns=list(COL_BY_CODE.values()))
        .reset_index())

wide = wide.merge(countries, on="country_iso2", how="left")  # attach names, ISO3, region, income group
unmatched = wide["country_iso3"].isna().sum()
print("Entity-years with no country metadata match:", int(unmatched))

meta_cols = ["country_name", "country_iso2", "country_iso3", "region", "income_group", "is_aggregate", "year"]
wide = (wide[meta_cols + list(COL_BY_CODE.values())]
        .sort_values(["year", "country_name"]).reset_index(drop=True))

for year, g in wide.groupby("year"):
    out = WB_YEARLY_DIR / f"worldbank_indicators_{year}.csv"
    try:
        g.to_csv(out, index=False)
    except PermissionError as exc:
        raise PermissionError(f"Cannot write {out}. Close it in Excel and rerun.") from exc
    print(f"{year}: {len(g)} rows, {g.shape[1]} columns")

panel_path = INTERIM_DIR / f"worldbank_wide_{START_YEAR}_{END_YEAR}.csv"
wide.to_csv(panel_path, index=False)
print("\nCombined panel:", panel_path, wide.shape)

Entity-years with no country metadata match: 0
2000: 266 rows, 37 columns
2001: 266 rows, 37 columns
2002: 266 rows, 37 columns
2003: 266 rows, 37 columns
2004: 266 rows, 37 columns
2005: 266 rows, 37 columns
2006: 266 rows, 37 columns
2007: 266 rows, 37 columns
2008: 266 rows, 37 columns
2009: 266 rows, 37 columns
2010: 266 rows, 37 columns
2011: 266 rows, 37 columns
2012: 266 rows, 37 columns
2013: 266 rows, 37 columns
2014: 266 rows, 37 columns
2015: 266 rows, 37 columns
2016: 266 rows, 37 columns
2017: 266 rows, 37 columns
2018: 266 rows, 37 columns
2019: 266 rows, 37 columns
2020: 266 rows, 37 columns
2021: 266 rows, 37 columns
2022: 266 rows, 37 columns
2023: 266 rows, 37 columns
2024: 266 rows, 37 columns
2025: 266 rows, 37 columns

Combined panel: d:\Pandu\IIT_Madras\DA2402\DA2402_Project_Earthquakes\interim\worldbank_wide_2000_2025.csv (6916, 37)


## 8. Coverage check

Shows which years and indicators are populated. Expect 2025 to be nearly empty and 2024 to be partial (WDI lags by about a year), so the income merge should use year t-1 or the latest available year, and the rule should be documented.

In [21]:
real = wide[~wide["is_aggregate"]]  # [real country-years, cols]
key = ["gdp_per_capita_current_usd", "gdp_per_capita_constant_2015_usd", "gdp_per_capita_ppp_constant_2021",
       "gni_per_capita_ppp_constant_2021", "population_total", "urban_population_pct",
       "electricity_access_pct", "health_expenditure_pct_gdp"]

print("Share of real countries with a value, by year (key indicators):")
display(real.groupby("year")[key].agg(lambda s: s.notna().mean()).round(2))

print("Overall non-null share per indicator (real countries):")
display(real[list(COL_BY_CODE.values())].notna().mean().sort_values().round(2).to_frame("non_null_share"))

Share of real countries with a value, by year (key indicators):


,gdp_per_capita_current_usd,gdp_per_capita_constant_2015_usd,gdp_per_capita_ppp_constant_2021,gni_per_capita_ppp_constant_2021,population_total,urban_population_pct,electricity_access_pct,health_expenditure_pct_gdp
year,,,,,,,,
2000,0.94,0.92,0.88,0.54,1.0,1.0,0.97,0.85
2001,0.94,0.92,0.88,0.55,1.0,1.0,0.97,0.85
2002,0.96,0.94,0.89,0.57,1.0,1.0,0.98,0.85
2003,0.96,0.94,0.89,0.57,1.0,1.0,0.98,0.86
2004,0.96,0.94,0.89,0.57,1.0,1.0,0.98,0.86
2005,0.96,0.94,0.89,0.58,1.0,1.0,0.98,0.86
2006,0.97,0.94,0.89,0.60,1.0,1.0,0.98,0.86
2007,0.97,0.94,0.90,0.61,1.0,1.0,0.99,0.86
2008,0.97,0.96,0.91,0.63,1.0,1.0,0.99,0.86


Overall non-null share per indicator (real countries):


,non_null_share
gini_index,0.33
slum_population_pct_urban,0.36
physicians_per_1000,0.51
hospital_beds_per_1000,0.52
adjusted_net_savings_pct_gni,0.54
pop_in_large_cities_pct,0.56
gni_per_capita_ppp_constant_2021,0.66
gni_per_capita_constant_2015_usd,0.67
health_expenditure_pct_gdp,0.81
inflation_cpi_pct,0.83


In [22]:
# Backward-compatible alias to the yearly CSV folder defined in Setup.
YEARLY_DIR = WB_YEARLY_DIR
print("Yearly World Bank CSV directory:", YEARLY_DIR)

Yearly World Bank CSV directory: d:\Pandu\IIT_Madras\DA2402\DA2402_Project_Earthquakes\interim\worldbank_yearly_csv
